# Learned-γ Arm 2 Diagnostic

## Motivation

The kinetic-rate regularisation experiment found that the **two-phase fine-tuned
run** (`kin_0.1_ft`) improved undamped geodesic compliance (cos 0.643→0.687,
$R^2$ −2.15→−1.82) but damped compliance stayed flat (cos ≈ 0.59). The damped
diagnostic adds a $-\gamma v$ correction using the *initial* $\gamma = 1.0$, but
after training the model's learned $\gamma$ is a free parameter (backed by
`raw_gamma` via softplus) that can drift substantially.

The kinetic-rate ratios of `kin_0.1_ft` are $[0.84, 0.67, 0.84, 0.92, 0.83]$,
corresponding to effective $\gamma \approx 0.17$–$0.39$ — far from 1.0.

## What this notebook does

1. Loads the six kinetic-rate checkpoints
2. Reads the **actual learned $\gamma$** from each model
3. Also estimates an **empirical $\gamma$** from the T-ratios (bypassing the
   model parameter, using the trajectory itself)
4. Re-runs Arm 2 geodesic compliance with:
   - $\gamma_{\text{init}} = 1.0$ (original)
   - $\gamma_{\text{learned}}$ from the model parameter
   - $\gamma_{\text{empirical}}$ from the T-ratio median
5. Compares all three to determine whether the damped-compliance gap is a
   measurement artifact or a genuine geometric failure

In [ ]:
# ── Cell 1: Environment setup ──────────────────────────────────────
import subprocess, sys, os, gc, math, json, time
from pathlib import Path
from dataclasses import fields as dc_fields

def run(cmd):
    print(f"$ {cmd}")
    subprocess.check_call(cmd, shell=True)

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    run('pip install -q transformers huggingface_hub pyarrow scipy scikit-learn')
    if not os.path.isdir('semsimula-paper'):
        run('git clone --depth 1 https://github.com/dimitarpg13/semsimula-paper.git')
    REPO = 'semsimula-paper'
else:
    REPO = os.environ.get('SEMSIMULA_PAPER', '.')

ARCH_DIR = os.path.join(REPO, 'notebooks', 'conservative_arch')
for p in [
    ARCH_DIR,
    os.path.join(ARCH_DIR, 'multixi'),
    os.path.join(ARCH_DIR, 'parf'),
    os.path.join(ARCH_DIR, 'energetic_minima'),
    os.path.join(ARCH_DIR, 'sarf_mass_variant'),
    os.path.join(ARCH_DIR, 'scaleup'),
]:
    if p not in sys.path:
        sys.path.insert(0, p)

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

DEVICE = (
    'cuda' if torch.cuda.is_available()
    else ('mps' if hasattr(torch.backends, 'mps')
          and torch.backends.mps.is_available() else 'cpu')
)
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    print('TF32 disabled for autograd.grad stability')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive/semsimula_gamma_diagnostic')
else:
    DRIVE_ROOT = Path.home() / 'semsimula_gamma_diagnostic'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
DRIVE_RESULTS = DRIVE_ROOT / 'results'
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print(f'Results dir: {DRIVE_RESULTS}')

In [ ]:
# ── Cell 2: Locate checkpoints ────────────────────────────────────
# ┌─────────────────────────────────────────────────────────────────┐
# │  CONFIGURE: edit these two lines to match your folder layout.  │
# │  GDRIVE_CKPT_DIR  — path under MyDrive (Colab)                 │
# │  LOCAL_CKPT_DIR   — absolute or ~ path (local runs)            │
# └─────────────────────────────────────────────────────────────────┘
GDRIVE_CKPT_DIR = 'semsimula_kinetic_rate/checkpoints'  # relative to MyDrive
LOCAL_CKPT_DIR  = '~/Downloads/semsimula_kinetic_rate/checkpoints'

if IN_COLAB:
    CKPT_DIR = Path('/content/drive/MyDrive') / GDRIVE_CKPT_DIR
else:
    CKPT_DIR = Path(LOCAL_CKPT_DIR).expanduser()

KEYS = ['baseline', 'kin_0.01', 'kin_0.1', 'kin_1.0', 'kin_0.1_noLN', 'kin_0.1_ft']
COLORS = {
    'baseline': 'tab:blue', 'kin_0.01': 'tab:cyan', 'kin_0.1': 'tab:orange',
    'kin_1.0': 'tab:green', 'kin_0.1_noLN': 'tab:red', 'kin_0.1_ft': 'tab:purple',
}

ckpt_paths = {}
for key in KEYS:
    p = CKPT_DIR / f'{key}.pt'
    assert p.exists(), f'Missing checkpoint: {p}'
    ckpt_paths[key] = p
    print(f'  {key}: {p}')

print(f'\n{len(ckpt_paths)} checkpoints found.')

In [ ]:
# ── Cell 3: Load data (OOM-safe lightweight path) ─────────────────
from data_module import get_batch, _download_hf_parquet, _resolve_tinystories_shard, _gpt2_tokenize
import pyarrow.parquet as pq

SCRIPTS_DIR = os.path.join(ARCH_DIR, 'scaleup')
LOGFREQ_PATH = os.path.join(SCRIPTS_DIR, 'results', 'logfreq_surprisal_tinystories.npy')

if not os.path.exists(LOGFREQ_PATH):
    print('Computing logfreq surprisal (one-time)...')
    os.makedirs(os.path.dirname(LOGFREQ_PATH), exist_ok=True)
    subprocess.run(
        [sys.executable, os.path.join(SCRIPTS_DIR, 'compute_unigram_frequencies_tinystories.py')],
        cwd=SCRIPTS_DIR, check=True,
    )

DATA_DIR = os.path.join(ARCH_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

val_cache = os.path.join(DATA_DIR, 'tinystories_val_only.npy')
if os.path.exists(val_cache):
    val_ids = np.load(val_cache)
    print(f'Loaded cached val tokens: {len(val_ids):,}')
else:
    val_fname = _resolve_tinystories_shard('data/validation-00000-of-00001')
    vp = _download_hf_parquet('roneneldan/TinyStories', val_fname, 'tinystories_val.parquet')
    val_texts = pq.read_table(vp, columns=['text'])['text'].to_pylist()
    val_ids = _gpt2_tokenize('\n\n'.join(val_texts[:2000]))
    del val_texts
    np.save(val_cache, val_ids)
    print(f'  Cached {len(val_ids):,} val tokens')

gc.collect()
print(f'Val tokens: {len(val_ids):,}')

In [ ]:
# ── Cell 4: Model loading + gamma extraction ──────────────────────
from model_multixi import (
    ScalarPotentialLMSARFMassLNMultiXi,
    SPLMSARFMassLNMultiXiConfig,
)


def load_model(key):
    ckpt = torch.load(ckpt_paths[key], map_location='cpu', weights_only=False)
    known = {f.name for f in dc_fields(SPLMSARFMassLNMultiXiConfig)}
    cfg = SPLMSARFMassLNMultiXiConfig(
        **{k: v for k, v in ckpt['config'].items() if k in known})
    if hasattr(cfg, 'logfreq_path'):
        cfg.logfreq_path = LOGFREQ_PATH
    model = ScalarPotentialLMSARFMassLNMultiXi(cfg)
    model.load_state_dict(ckpt['model_state_dict'], strict=False)
    model.to(DEVICE).eval()
    return model, ckpt


def extract_gamma_info(model, ckpt):
    """Extract the learned gamma and compute empirical gamma from T ratios."""
    gamma_learned = model.gamma.item()

    # Empirical gamma from the final T-ratios logged during training
    kin_profiles = ckpt.get('kin_profiles', [])
    if kin_profiles and kin_profiles[-1].get('T_ratios'):
        ratios = kin_profiles[-1]['T_ratios']
        valid_ratios = [r for r in ratios if r > 0]
        if valid_ratios:
            median_ratio = float(np.median(valid_ratios))
            gamma_empirical = -math.log(max(median_ratio, 1e-12))
        else:
            gamma_empirical = gamma_learned
    else:
        gamma_empirical = gamma_learned

    return gamma_learned, gamma_empirical


# Extract gamma values from all checkpoints
print('═' * 70)
print('GAMMA VALUES PER CHECKPOINT')
print('═' * 70)
print(f'{"Key":<20} {"γ_init":>8} {"γ_learned":>10} {"γ_empirical":>12} {"T_ratios (final)"}')
print('-' * 70)

gamma_info = {}
for key in KEYS:
    model, ckpt = load_model(key)
    g_learned, g_empirical = extract_gamma_info(model, ckpt)
    gamma_info[key] = {'learned': g_learned, 'empirical': g_empirical}

    profiles = ckpt.get('kin_profiles', [])
    if profiles and profiles[-1].get('T_ratios'):
        ratio_str = '[' + ', '.join(f'{r:.3f}' for r in profiles[-1]['T_ratios']) + ']'
    else:
        ratio_str = 'N/A'
    print(f'{key:<20} {1.0:>8.3f} {g_learned:>10.4f} {g_empirical:>12.4f}   {ratio_str}')

    model.cpu(); del model, ckpt
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print('═' * 70)

In [ ]:
# ── Cell 5: Arm 2 — Multi-gamma geodesic compliance ───────────────

N_EVAL_BATCHES = 3
BLOCK_SIZE = 128
EVAL_BS = 4
rng_diag = np.random.default_rng(123)
eval_batches = []
for _ in range(N_EVAL_BATCHES):
    xb, _ = get_batch(val_ids, EVAL_BS, BLOCK_SIZE, rng_diag)
    eval_batches.append(torch.tensor(xb))


def extract_trajectory(model, x):
    with torch.enable_grad():
        out = model(x, targets=None, return_trajectory=True, return_xi_trajectory=False)
        logits, _loss, traj = out[0], out[1], out[2]
    if DEVICE == 'cuda': torch.cuda.synchronize()
    traj_cpu = [h.detach().cpu() for h in traj]
    del traj, out, logits
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return traj_cpu


def compute_grad_V(model, h):
    h_in = h.detach().requires_grad_(True)
    xis = model.xi_module(h_in.detach())
    V = model.V_theta(xis, h_in)
    return torch.autograd.grad(V.sum(), h_in, create_graph=False)[0].detach()


def get_mass(model, x):
    emb = model._embed(x)
    m = model.compute_mass(x, emb)
    return m.detach().cpu() if isinstance(m, torch.Tensor) else m


def arm2_with_gamma(model, gamma_val, eval_batches):
    """Run Arm 2 compliance using a specific gamma value."""
    comp_und, comp_dmp, cos_und, cos_dmp = None, None, None, None

    for bi in range(len(eval_batches)):
        x = eval_batches[bi].to(DEVICE)
        traj = extract_trajectory(model, x)
        L = len(traj) - 1

        if comp_und is None:
            comp_und = [[] for _ in range(L - 1)]
            comp_dmp = [[] for _ in range(L - 1)]
            cos_und  = [[] for _ in range(L - 1)]
            cos_dmp  = [[] for _ in range(L - 1)]

        with torch.no_grad():
            m = get_mass(model, x)
            m_flat = m.squeeze(-1) if isinstance(m, torch.Tensor) and m.dim() > 2 else m
        del x

        for ell in range(1, L):
            h_prev = traj[ell - 1].to(DEVICE)
            h_curr = traj[ell].to(DEVICE)
            h_next = traj[ell + 1].to(DEVICE)
            v = h_curr - h_prev
            a_obs = h_next - 2 * h_curr + h_prev
            del h_prev, h_next

            grad_V = compute_grad_V(model, h_curr)
            v_norm2 = (v ** 2).sum(dim=-1, keepdim=True)
            mf = (m_flat.unsqueeze(-1).to(DEVICE)
                  if isinstance(m_flat, torch.Tensor) and m_flat.dim() == 2 else m_flat)
            denom = (2.0 * 0.5 * mf * v_norm2).clamp(min=1e-8)
            gV_dot_v = (grad_V * v).sum(dim=-1, keepdim=True)
            a_j = (2.0 * v * gV_dot_v - v_norm2 * grad_V) / denom
            a_d = a_j - gamma_val * v

            a2 = (a_obs ** 2).sum(dim=-1).mean().item()
            comp_und[ell-1].append(
                1.0 - ((a_obs - a_j) ** 2).sum(dim=-1).mean().item() / max(a2, 1e-12))
            comp_dmp[ell-1].append(
                1.0 - ((a_obs - a_d) ** 2).sum(dim=-1).mean().item() / max(a2, 1e-12))
            af = a_obs.reshape(-1, a_obs.shape[-1])
            cos_und[ell-1].append(
                F.cosine_similarity(af, a_j.reshape(-1, a_j.shape[-1]), dim=-1).mean().item())
            cos_dmp[ell-1].append(
                F.cosine_similarity(af, a_d.reshape(-1, a_d.shape[-1]), dim=-1).mean().item())
            del h_curr, v, a_obs, grad_V, a_j, a_d

        del traj
        if DEVICE == 'cuda': torch.cuda.empty_cache()

    return {
        'r2_und':      [float(np.mean(l)) for l in comp_und],
        'r2_dmp':      [float(np.mean(l)) for l in comp_dmp],
        'cos_und':     [float(np.mean(l)) for l in cos_und],
        'cos_dmp':     [float(np.mean(l)) for l in cos_dmp],
        'r2_und_mean': float(np.mean([np.mean(l) for l in comp_und])),
        'r2_dmp_mean': float(np.mean([np.mean(l) for l in comp_dmp])),
        'cos_und_mean': float(np.mean([np.mean(l) for l in cos_und])),
        'cos_dmp_mean': float(np.mean([np.mean(l) for l in cos_dmp])),
    }


print('═' * 70)
print('ARM 2: Multi-Gamma Geodesic Compliance')
print('═' * 70)

all_results = {}

for key in KEYS:
    print(f'\n── {key} ──')
    model, ckpt = load_model(key)
    gi = gamma_info[key]
    gammas = {
        'init_1.0':  1.0,
        'learned':   gi['learned'],
        'empirical': gi['empirical'],
    }
    key_results = {}
    for gname, gval in gammas.items():
        print(f'  γ={gval:.4f} ({gname})...', end=' ', flush=True)
        a2 = arm2_with_gamma(model, gval, eval_batches)
        a2['gamma_value'] = gval
        a2['gamma_source'] = gname
        key_results[gname] = a2
        print(f'cos(dmp)={a2["cos_dmp_mean"]:.4f}  R²(dmp)={a2["r2_dmp_mean"]:.4f}')

    all_results[key] = key_results
    model.cpu(); del model, ckpt
    gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

print('\n✓ Arm 2 multi-gamma complete.')

In [ ]:
# ── Cell 6: Summary table ─────────────────────────────────────────

print('\n' + '═' * 110)
print('MULTI-GAMMA ARM 2 SUMMARY')
print('═' * 110)
print(f'{"Key":<18} {"γ source":<12} {"γ value":>8} '
      f'{"cos(und)":>10} {"R²(und)":>10} '
      f'{"cos(dmp)":>10} {"R²(dmp)":>10} '
      f'{"Δcos vs γ=1":>12}')
print('-' * 110)

for key in KEYS:
    kr = all_results[key]
    cos_init = kr['init_1.0']['cos_dmp_mean']
    for gname in ['init_1.0', 'learned', 'empirical']:
        a2 = kr[gname]
        delta = a2['cos_dmp_mean'] - cos_init
        delta_str = f'{delta:+.4f}' if gname != 'init_1.0' else '—'
        print(f'{key:<18} {gname:<12} {a2["gamma_value"]:>8.4f} '
              f'{a2["cos_und_mean"]:>10.4f} {a2["r2_und_mean"]:>10.4f} '
              f'{a2["cos_dmp_mean"]:>10.4f} {a2["r2_dmp_mean"]:>10.4f} '
              f'{delta_str:>12}')
    print()

print('═' * 110)
print()
print('Key questions answered:')
print('  1. Does using γ_learned instead of γ=1.0 improve damped compliance?')
print('  2. Is the damped gap a measurement artifact (wrong γ) or genuine?')
print('  3. Does γ_empirical (from T-ratios) differ from γ_learned (model param)?')

In [ ]:
# ── Cell 7: Comparison plots ──────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(17, 10))

for ax, key in zip(axes.flat, KEYS):
    kr = all_results[key]
    for gname, ls, marker in [('init_1.0', '--', 'o'), ('learned', '-', 's'), ('empirical', ':', '^')]:
        a2 = kr[gname]
        layers = range(1, len(a2['cos_dmp']) + 1)
        label = f'γ={a2["gamma_value"]:.3f} ({gname})'
        ax.plot(layers, a2['cos_dmp'], ls=ls, marker=marker, markersize=4,
                label=label, alpha=0.85)
    ax.set_title(f'{key}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Layer', fontsize=8)
    ax.set_ylabel('Cosine (damped)', fontsize=8)
    ax.legend(fontsize=6, loc='lower left')
    ax.axhline(0, color='gray', ls=':', lw=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Damped Cosine Compliance: γ_init=1.0 vs γ_learned vs γ_empirical',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(DRIVE_RESULTS / 'gamma_cosine_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

# R² comparison
fig2, axes2 = plt.subplots(2, 3, figsize=(17, 10))

for ax, key in zip(axes2.flat, KEYS):
    kr = all_results[key]
    for gname, ls, marker in [('init_1.0', '--', 'o'), ('learned', '-', 's'), ('empirical', ':', '^')]:
        a2 = kr[gname]
        layers = range(1, len(a2['r2_dmp']) + 1)
        label = f'γ={a2["gamma_value"]:.3f} ({gname})'
        ax.plot(layers, a2['r2_dmp'], ls=ls, marker=marker, markersize=4,
                label=label, alpha=0.85)
    ax.set_title(f'{key}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Layer', fontsize=8)
    ax.set_ylabel('R² (damped)', fontsize=8)
    ax.legend(fontsize=6, loc='lower left')
    ax.axhline(0, color='gray', ls=':', lw=0.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.suptitle('Damped R² Compliance: γ_init=1.0 vs γ_learned vs γ_empirical',
             fontweight='bold', fontsize=13, y=1.02)
plt.tight_layout()
fig2.savefig(DRIVE_RESULTS / 'gamma_r2_comparison.png', dpi=150,
            bbox_inches='tight', facecolor='white')
plt.show()

print(f'\nSaved: {DRIVE_RESULTS / "gamma_cosine_comparison.png"}')
print(f'Saved: {DRIVE_RESULTS / "gamma_r2_comparison.png"}')

In [ ]:
# ── Cell 8: Focused kin_0.1_ft analysis ───────────────────────────
# The two-phase run is the most important — show its per-layer profile
# for all three gamma values side by side.

key = 'kin_0.1_ft'
kr = all_results[key]

fig3, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for gname, ls, marker, color in [
    ('init_1.0',  '--', 'o', 'tab:red'),
    ('learned',   '-',  's', 'tab:blue'),
    ('empirical', ':',  '^', 'tab:green'),
]:
    a2 = kr[gname]
    layers = range(1, len(a2['cos_dmp']) + 1)
    label = f'γ={a2["gamma_value"]:.3f} ({gname})'
    ax1.plot(layers, a2['cos_dmp'], ls=ls, marker=marker, markersize=6,
             color=color, label=label, lw=2)
    ax2.plot(layers, a2['r2_dmp'], ls=ls, marker=marker, markersize=6,
             color=color, label=label, lw=2)

# Also show undamped as reference
a2_und = kr['init_1.0']
layers = range(1, len(a2_und['cos_und']) + 1)
ax1.plot(layers, a2_und['cos_und'], ls='-', marker='D', markersize=4,
         color='gray', alpha=0.5, label='undamped', lw=1.5)
ax2.plot(layers, a2_und['r2_und'], ls='-', marker='D', markersize=4,
         color='gray', alpha=0.5, label='undamped', lw=1.5)

ax1.set_title('kin_0.1_ft: Cosine Compliance', fontweight='bold')
ax2.set_title('kin_0.1_ft: R² Magnitude Compliance', fontweight='bold')
for ax in (ax1, ax2):
    ax.set_xlabel('Layer'); ax.axhline(0, color='gray', ls=':', lw=0.5)
    ax.legend(fontsize=8); ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
ax1.set_ylabel('Cosine'); ax2.set_ylabel('R²')

plt.suptitle('Two-Phase Fine-Tuned Model: Effect of γ on Compliance',
             fontweight='bold', y=1.02)
plt.tight_layout()
fig3.savefig(DRIVE_RESULTS / 'kin_ft_gamma_detail.png', dpi=150,
             bbox_inches='tight', facecolor='white')
plt.show()
print(f'Saved: {DRIVE_RESULTS / "kin_ft_gamma_detail.png"}')

In [ ]:
# ── Cell 9: Persist full results ──────────────────────────────────

report = {
    'gamma_info': gamma_info,
    'arm2_multi_gamma': {},
}
for key in KEYS:
    report['arm2_multi_gamma'][key] = all_results[key]

report_path = DRIVE_RESULTS / 'gamma_diagnostic_report.json'
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2, default=str)

print(f'Full results: {report_path}')
print('\n✓ Gamma diagnostic experiment complete.')